# BUSI Notebook 1/3: Main Model + Baselines + Evaluation

**Run this notebook first.** It trains the main multi-task model and two
baselines, evaluates them, and saves checkpoints + results you'll need
for Notebooks 2 and 3.

**Recommended:** Runtime -> Change runtime type -> GPU.

**After this finishes:** download the whole `outputs/` folder
(right-click it in the Colab file browser -> Download) — you'll upload
the checkpoint files from it into Notebooks 2 and 3.


## 1. Setup

In [ ]:
!pip install -q torch torchvision pillow numpy matplotlib seaborn scikit-learn

In [ ]:
import os, re, json, zipfile, random
from glob import glob
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 2. Load the dataset

Set `DATA_ZIP_PATH` below, or leave as-is and upload when prompted.


In [ ]:
DATA_ZIP_PATH = "archive.zip"
EXTRACT_DIR = "busi_data"

if not os.path.exists(DATA_ZIP_PATH):
    try:
        from google.colab import files
        print(f"'{DATA_ZIP_PATH}' not found. Please upload the BUSI dataset zip file:")
        uploaded = files.upload()
        DATA_ZIP_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"'{DATA_ZIP_PATH}' not found and not running in Colab.")

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"Extracted to {EXTRACT_DIR}/")
else:
    print(f"{EXTRACT_DIR}/ already exists, skipping extraction.")

candidates = glob(os.path.join(EXTRACT_DIR, "**", "Dataset_BUSI_with_GT"), recursive=True)
if not candidates:
    if all(os.path.isdir(os.path.join(EXTRACT_DIR, c)) for c in ["benign", "malignant", "normal"]):
        DATA_ROOT = EXTRACT_DIR
    else:
        raise FileNotFoundError("Could not locate Dataset_BUSI_with_GT folder inside the extracted zip.")
else:
    DATA_ROOT = candidates[0]

print("Dataset root:", DATA_ROOT)
print("Classes found:", os.listdir(DATA_ROOT))


## 3. Index the dataset

In [ ]:
CLASS_NAMES = ["benign", "malignant", "normal"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

def index_dataset(root_dir):
    records = []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(root_dir, cls)
        if not os.path.isdir(cls_dir):
            raise FileNotFoundError(f"Expected class folder not found: {cls_dir}")
        all_files = os.listdir(cls_dir)
        image_files = sorted(f for f in all_files if "_mask" not in f)
        for img_file in image_files:
            stem = os.path.splitext(img_file)[0]
            mask_pattern = re.compile(rf"^{re.escape(stem)}_mask(_\d+)?\.png$")
            masks = sorted(os.path.join(cls_dir, f) for f in all_files if mask_pattern.match(f))
            records.append({"image": os.path.join(cls_dir, img_file), "masks": masks,
                             "label": CLASS_TO_IDX[cls], "class": cls})
    return records

records = index_dataset(DATA_ROOT)
print(f"Total records: {len(records)}")
counts = defaultdict(int)
for r in records:
    counts[r["class"]] += 1
print("Class distribution:", dict(counts))


## 4. Stratified train / val / test split

**Important:** this uses a fixed `seed=42`. Notebooks 2 and 3 use the
exact same indexing + split code with the same seed, so as long as you
upload the same dataset zip, the splits will match exactly — this keeps
your test set consistent across all three notebooks.


In [ ]:
def stratified_split(records, val_frac=0.15, test_frac=0.15, seed=SEED):
    rng = np.random.RandomState(seed)
    by_class = defaultdict(list)
    for r in records:
        by_class[r["class"]].append(r)
    train, val, test = [], [], []
    for cls, items in by_class.items():
        items = items.copy()
        rng.shuffle(items)
        n = len(items)
        n_val = int(round(n * val_frac))
        n_test = int(round(n * test_frac))
        test.extend(items[:n_test])
        val.extend(items[n_test:n_test + n_val])
        train.extend(items[n_test + n_val:])
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

train_recs, val_recs, test_recs = stratified_split(records)
print(f"Train: {len(train_recs)}  Val: {len(val_recs)}  Test: {len(test_recs)}")
for name, split in [("train", train_recs), ("val", val_recs), ("test", test_recs)]:
    c = defaultdict(int)
    for r in split:
        c[r["class"]] += 1
    print(f"  {name}: {dict(c)}")

os.makedirs("outputs", exist_ok=True)
with open("outputs/splits.json", "w") as f:
    json.dump({"train": [r["image"] for r in train_recs],
               "val": [r["image"] for r in val_recs],
               "test": [r["image"] for r in test_recs]}, f, indent=2)


## 5. PyTorch Dataset

In [ ]:
IMG_SIZE = 256

def load_merged_mask(mask_paths, size=None):
    if len(mask_paths) == 0:
        return None
    from PIL import Image
    merged = None
    for p in mask_paths:
        m = np.array(Image.open(p).convert("L"))
        m = (m > 127).astype(np.uint8)
        merged = m if merged is None else np.maximum(merged, m)
    mask_img = Image.fromarray(merged * 255)
    if size is not None:
        mask_img = mask_img.resize(size, resample=Image.NEAREST)
    return mask_img

class BUSIDataset(Dataset):
    def __init__(self, records, img_size=256, augment=False):
        self.records = records
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        from PIL import Image
        rec = self.records[idx]
        size = (self.img_size, self.img_size)
        image = Image.open(rec["image"]).convert("RGB").resize(size, resample=Image.BILINEAR)
        mask_img = load_merged_mask(rec["masks"], size=size)
        if mask_img is None:
            mask_img = Image.new("L", size, 0)
        if self.augment:
            image, mask_img = self._augment(image, mask_img)
        image_t = TF.to_tensor(image)
        mask_t = TF.to_tensor(mask_img)
        mask_t = (mask_t > 0.5).float()
        label_t = torch.tensor(rec["label"], dtype=torch.long)
        return image_t, mask_t, label_t

    def _augment(self, image, mask):
        if np.random.rand() < 0.5:
            image = TF.hflip(image); mask = TF.hflip(mask)
        if np.random.rand() < 0.5:
            image = TF.vflip(image); mask = TF.vflip(mask)
        if np.random.rand() < 0.5:
            angle = np.random.uniform(-15, 15)
            image = TF.rotate(image, angle, interpolation=TF.InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, angle, interpolation=TF.InterpolationMode.NEAREST)
        return image, mask

BATCH_SIZE = 16
train_ds = BUSIDataset(train_recs, img_size=IMG_SIZE, augment=True)
val_ds = BUSIDataset(val_recs, img_size=IMG_SIZE, augment=False)
test_ds = BUSIDataset(test_recs, img_size=IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("Batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))


## 6. Visualize a few samples (sanity check)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    img, mask, label = train_ds[i]
    axes[0, i].imshow(img.permute(1, 2, 0))
    axes[0, i].set_title(CLASS_NAMES[label.item()]); axes[0, i].axis("off")
    axes[1, i].imshow(mask.squeeze(), cmap="gray")
    axes[1, i].set_title("mask"); axes[1, i].axis("off")
plt.tight_layout(); plt.show()


## 7. Model building blocks: CBAM attention

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Linear(channels, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, channels))

    def forward(self, x):
        avg = F.adaptive_avg_pool2d(x, 1).flatten(1)
        mx = F.adaptive_max_pool2d(x, 1).flatten(1)
        attn = torch.sigmoid(self.mlp(avg) + self.mlp(mx))
        return x * attn.unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        attn = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.attn = CBAM(out_channels)

    def forward(self, x, skip):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return self.attn(x)


## 8. Main model: Multi-Task Network (segmentation + mask-guided classification)

In [ ]:
class BUSIMultiTaskNet(nn.Module):
    def __init__(self, num_classes=3, pretrained=True):
        super().__init__()
        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = tvm.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.final_upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.seg_head = nn.Conv2d(32, 1, kernel_size=1)
        self.cls_dropout = nn.Dropout(0.3)
        self.cls_fc = nn.Linear(512, num_classes)

    def forward(self, x):
        input_size = x.shape[-2:]
        s0 = self.stem(x); p0 = self.pool(s0)
        s1 = self.enc1(p0); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        d4 = self.dec4(s4, s3); d3 = self.dec3(d4, s2); d2 = self.dec2(d3, s1); d1 = self.dec1(d2, s0)
        d0 = self.final_upsample(d1)
        if d0.shape[-2:] != input_size:
            d0 = F.interpolate(d0, size=input_size, mode="bilinear", align_corners=False)
        seg_logits = self.seg_head(d0)
        mask_prob = torch.sigmoid(seg_logits.detach())
        mask_small = F.adaptive_avg_pool2d(mask_prob, s4.shape[-2:])
        weighted_feats = s4 * (1.0 + mask_small)
        pooled = F.adaptive_avg_pool2d(weighted_feats, 1).flatten(1)
        cls_logits = self.cls_fc(self.cls_dropout(pooled))
        return seg_logits, cls_logits


## 9. Baseline models (for comparison)

In [ ]:
class BaselineUNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = tvm.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4

        def plain_block(in_c, skip_c, out_c):
            return nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)), nn.Sequential(
                nn.Conv2d(in_c + skip_c, out_c, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
        self.up4, self.conv4 = plain_block(512, 256, 256)
        self.up3, self.conv3 = plain_block(256, 128, 128)
        self.up2, self.conv2 = plain_block(128, 64, 64)
        self.up1, self.conv1 = plain_block(64, 64, 32)
        self.final_upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.seg_head = nn.Conv2d(32, 1, kernel_size=1)

    def _dec(self, up, conv, x, skip):
        x = up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return conv(torch.cat([x, skip], dim=1))

    def forward(self, x):
        input_size = x.shape[-2:]
        s0 = self.stem(x); p0 = self.pool(s0)
        s1 = self.enc1(p0); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        d4 = self._dec(self.up4, self.conv4, s4, s3)
        d3 = self._dec(self.up3, self.conv3, d4, s2)
        d2 = self._dec(self.up2, self.conv2, d3, s1)
        d1 = self._dec(self.up1, self.conv1, d2, s0)
        d0 = self.final_upsample(d1)
        if d0.shape[-2:] != input_size:
            d0 = F.interpolate(d0, size=input_size, mode="bilinear", align_corners=False)
        return self.seg_head(d0)


class BaselineResNetClassifier(nn.Module):
    def __init__(self, num_classes=3, pretrained=True):
        super().__init__()
        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = tvm.resnet34(weights=weights)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)


## 10. Losses and metrics

In [ ]:
def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits).flatten(1)
    targets = targets.flatten(1)
    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    return 1 - ((2 * intersection + eps) / (union + eps)).mean()

def dice_score(logits, targets, threshold=0.5, eps=1e-6):
    with torch.no_grad():
        preds = (torch.sigmoid(logits) > threshold).float().flatten(1)
        targets = targets.flatten(1)
        intersection = (preds * targets).sum(dim=1)
        union = preds.sum(dim=1) + targets.sum(dim=1)
        return ((2 * intersection + eps) / (union + eps)).mean().item()

def iou_score(logits, targets, threshold=0.5, eps=1e-6):
    with torch.no_grad():
        preds = (torch.sigmoid(logits) > threshold).float().flatten(1)
        targets = targets.flatten(1)
        intersection = (preds * targets).sum(dim=1)
        union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
        return ((intersection + eps) / (union + eps)).mean().item()

def compute_class_weights(records, num_classes):
    counts = np.zeros(num_classes)
    for r in records:
        counts[r["label"]] += 1
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

class CombinedLoss(nn.Module):
    def __init__(self, seg_weight=1.0, cls_weight=1.0, class_weights=None):
        super().__init__()
        self.seg_weight = seg_weight
        self.cls_weight = cls_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def forward(self, seg_logits, seg_target, cls_logits, cls_target):
        seg_loss = self.bce(seg_logits, seg_target) + dice_loss(seg_logits, seg_target)
        cls_loss = self.ce(cls_logits, cls_target)
        total = self.seg_weight * seg_loss + self.cls_weight * cls_loss
        return total, seg_loss.item(), cls_loss.item()

class_weights = compute_class_weights(train_recs, len(CLASS_NAMES)).to(DEVICE)
print("Class weights (benign, malignant, normal):", class_weights.tolist())


## 11. Train the main multi-task model

In [ ]:
EPOCHS = 40
LR = 1e-4
PRETRAINED = True

def run_multitask_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss = total_dice = total_iou = 0.0
    correct = n_samples = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, masks, labels in loader:
            images, masks, labels = images.to(device), masks.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            seg_logits, cls_logits = model(images)
            loss, seg_l, cls_l = criterion(seg_logits, masks, cls_logits, labels)
            if train:
                loss.backward(); optimizer.step()
            bs = images.size(0)
            total_loss += loss.item() * bs
            total_dice += dice_score(seg_logits, masks) * bs
            total_iou += iou_score(seg_logits, masks) * bs
            correct += (cls_logits.argmax(dim=1) == labels).sum().item()
            n_samples += bs
    return {"loss": total_loss / n_samples, "dice": total_dice / n_samples,
            "iou": total_iou / n_samples, "accuracy": correct / n_samples}

model = BUSIMultiTaskNet(num_classes=len(CLASS_NAMES), pretrained=PRETRAINED).to(DEVICE)
criterion = CombinedLoss(class_weights=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

best_score = -1.0
history = []
for epoch in range(1, EPOCHS + 1):
    tr = run_multitask_epoch(model, train_loader, criterion, optimizer, DEVICE, train=True)
    va = run_multitask_epoch(model, val_loader, criterion, optimizer, DEVICE, train=False)
    scheduler.step(va["loss"])
    combined = va["dice"] + va["accuracy"]
    is_best = combined > best_score
    if is_best:
        best_score = combined
        torch.save(model.state_dict(), "outputs/best_multitask_model.pt")
    print(f"Epoch {epoch:3d}/{EPOCHS} | train_loss {tr['loss']:.4f} | val_loss {va['loss']:.4f} | "
          f"val_dice {va['dice']:.4f} | val_acc {va['accuracy']:.4f}" + ("  <- best" if is_best else ""))
    history.append({"epoch": epoch, "train": tr, "val": va})

with open("outputs/history.json", "w") as f:
    json.dump(history, f, indent=2)


## 12. Train baseline models

In [ ]:
def run_seg_only_epoch(model, loader, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss = total_dice = total_iou = n_samples = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, masks, _ in loader:
            images, masks = images.to(device), masks.to(device)
            if train:
                optimizer.zero_grad()
            logits = model(images)
            loss = nn.functional.binary_cross_entropy_with_logits(logits, masks) + dice_loss(logits, masks)
            if train:
                loss.backward(); optimizer.step()
            bs = images.size(0)
            total_loss += loss.item() * bs
            total_dice += dice_score(logits, masks) * bs
            total_iou += iou_score(logits, masks) * bs
            n_samples += bs
    return {"loss": total_loss / n_samples, "dice": total_dice / n_samples, "iou": total_iou / n_samples}

def run_cls_only_epoch(model, loader, optimizer, device, class_weights, train=True):
    model.train() if train else model.eval()
    ce = nn.CrossEntropyLoss(weight=class_weights)
    total_loss = correct = n_samples = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, _, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            logits = model(images)
            loss = ce(logits, labels)
            if train:
                loss.backward(); optimizer.step()
            bs = images.size(0)
            total_loss += loss.item() * bs
            correct += (logits.argmax(dim=1) == labels).sum().item()
            n_samples += bs
    return {"loss": total_loss / n_samples, "accuracy": correct / n_samples}

baseline_unet = BaselineUNet(pretrained=PRETRAINED).to(DEVICE)
opt_unet = torch.optim.AdamW(baseline_unet.parameters(), lr=LR, weight_decay=1e-4)
best_unet_dice = -1.0
for epoch in range(1, EPOCHS + 1):
    tr = run_seg_only_epoch(baseline_unet, train_loader, opt_unet, DEVICE, train=True)
    va = run_seg_only_epoch(baseline_unet, val_loader, opt_unet, DEVICE, train=False)
    if va["dice"] > best_unet_dice:
        best_unet_dice = va["dice"]
        torch.save(baseline_unet.state_dict(), "outputs/best_baseline_unet.pt")
    if epoch % 5 == 0 or epoch == 1:
        print(f"[Baseline U-Net] Epoch {epoch}/{EPOCHS} | val_dice {va['dice']:.4f}")

baseline_resnet = BaselineResNetClassifier(num_classes=len(CLASS_NAMES), pretrained=PRETRAINED).to(DEVICE)
opt_resnet = torch.optim.AdamW(baseline_resnet.parameters(), lr=LR, weight_decay=1e-4)
best_resnet_acc = -1.0
for epoch in range(1, EPOCHS + 1):
    tr = run_cls_only_epoch(baseline_resnet, train_loader, opt_resnet, DEVICE, class_weights, train=True)
    va = run_cls_only_epoch(baseline_resnet, val_loader, opt_resnet, DEVICE, class_weights, train=False)
    if va["accuracy"] > best_resnet_acc:
        best_resnet_acc = va["accuracy"]
        torch.save(baseline_resnet.state_dict(), "outputs/best_baseline_resnet.pt")
    if epoch % 5 == 0 or epoch == 1:
        print(f"[Baseline ResNet] Epoch {epoch}/{EPOCHS} | val_acc {va['accuracy']:.4f}")

print("\nBaseline training done.")


## 13. Final evaluation on the held-out test set

In [ ]:
model.load_state_dict(torch.load("outputs/best_multitask_model.pt", map_location=DEVICE))
baseline_unet.load_state_dict(torch.load("outputs/best_baseline_unet.pt", map_location=DEVICE))
baseline_resnet.load_state_dict(torch.load("outputs/best_baseline_resnet.pt", map_location=DEVICE))
model.eval(); baseline_unet.eval(); baseline_resnet.eval()

all_labels, all_preds_mt, all_preds_resnet = [], [], []
all_probs_mt = []
dice_mt, iou_mt, dice_unet, iou_unet = [], [], [], []

with torch.no_grad():
    for images, masks, labels in test_loader:
        images, masks, labels = images.to(DEVICE), masks.to(DEVICE), labels.to(DEVICE)
        seg_logits, cls_logits = model(images)
        dice_mt.append(dice_score(seg_logits, masks))
        iou_mt.append(iou_score(seg_logits, masks))
        preds_mt = cls_logits.argmax(dim=1)
        all_preds_mt.extend(preds_mt.cpu().numpy())
        all_probs_mt.extend(torch.softmax(cls_logits, dim=1).cpu().numpy())

        unet_logits = baseline_unet(images)
        dice_unet.append(dice_score(unet_logits, masks))
        iou_unet.append(iou_score(unet_logits, masks))

        resnet_logits = baseline_resnet(images)
        all_preds_resnet.extend(resnet_logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_labels = np.array(all_labels)
all_preds_mt = np.array(all_preds_mt)
all_preds_resnet = np.array(all_preds_resnet)
all_probs_mt = np.array(all_probs_mt)

results = {
    "Multi-Task Model": {"Dice": float(np.mean(dice_mt)), "IoU": float(np.mean(iou_mt)),
                          "Accuracy": float(np.mean(all_preds_mt == all_labels)),
                          "Macro-F1": float(f1_score(all_labels, all_preds_mt, average="macro"))},
    "Baseline U-Net (seg only)": {"Dice": float(np.mean(dice_unet)), "IoU": float(np.mean(iou_unet)),
                                   "Accuracy": None, "Macro-F1": None},
    "Baseline ResNet34 (cls only)": {"Dice": None, "IoU": None,
                                      "Accuracy": float(np.mean(all_preds_resnet == all_labels)),
                                      "Macro-F1": float(f1_score(all_labels, all_preds_resnet, average="macro"))},
}

print("=" * 70)
print(f"{'Model':<30}{'Dice':<10}{'IoU':<10}{'Accuracy':<12}{'Macro-F1':<10}")
print("=" * 70)
for name, m in results.items():
    def fmt(v): return f"{v:.4f}" if v is not None else "-"
    print(f"{name:<30}{fmt(m['Dice']):<10}{fmt(m['IoU']):<10}{fmt(m['Accuracy']):<12}{fmt(m['Macro-F1']):<10}")

with open("outputs/test_results_table.json", "w") as f:
    json.dump(results, f, indent=2)


## 14. Plots: training curves, confusion matrix, sample predictions

In [ ]:
epochs_range = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs_range, [h["train"]["loss"] for h in history], label="train")
axes[0].plot(epochs_range, [h["val"]["loss"] for h in history], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(epochs_range, [h["train"]["dice"] for h in history], label="train")
axes[1].plot(epochs_range, [h["val"]["dice"] for h in history], label="val")
axes[1].set_title("Segmentation Dice"); axes[1].set_xlabel("Epoch"); axes[1].legend()
axes[2].plot(epochs_range, [h["train"]["accuracy"] for h in history], label="train")
axes[2].plot(epochs_range, [h["val"]["accuracy"] for h in history], label="val")
axes[2].set_title("Classification Accuracy"); axes[2].set_xlabel("Epoch"); axes[2].legend()
plt.tight_layout(); plt.savefig("outputs/training_curves.png", dpi=150); plt.show()


In [ ]:
cm = confusion_matrix(all_labels, all_preds_mt)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — Multi-Task Model (Test Set)")
plt.tight_layout(); plt.savefig("outputs/confusion_matrix.png", dpi=150); plt.show()


In [ ]:
model.eval()
n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(9, 3 * n_show))
shown = 0
with torch.no_grad():
    for images, masks, labels in test_loader:
        for i in range(images.size(0)):
            if shown >= n_show:
                break
            img = images[i:i+1].to(DEVICE)
            seg_logits, cls_logits = model(img)
            pred_mask = (torch.sigmoid(seg_logits) > 0.5).float().cpu().squeeze()
            pred_label = CLASS_NAMES[cls_logits.argmax(dim=1).item()]
            true_label = CLASS_NAMES[labels[i].item()]
            axes[shown, 0].imshow(images[i].permute(1, 2, 0))
            axes[shown, 0].set_title(f"True: {true_label} | Pred: {pred_label}"); axes[shown, 0].axis("off")
            axes[shown, 1].imshow(masks[i].squeeze(), cmap="gray")
            axes[shown, 1].set_title("Ground truth mask"); axes[shown, 1].axis("off")
            axes[shown, 2].imshow(pred_mask, cmap="gray")
            axes[shown, 2].set_title("Predicted mask"); axes[shown, 2].axis("off")
            shown += 1
        if shown >= n_show:
            break
plt.tight_layout(); plt.savefig("outputs/sample_predictions.png", dpi=150); plt.show()


## Done — Notebook 1

Files saved to `outputs/`:
- `best_multitask_model.pt`, `best_baseline_unet.pt`, `best_baseline_resnet.pt` — **upload these into Notebook 2**
- `splits.json` — the exact train/val/test split (Notebooks 2 & 3 reproduce it from the same seed + zip)
- `history.json`, `test_results_table.json`
- `training_curves.png`, `confusion_matrix.png`, `sample_predictions.png`

**Next:** download the `outputs/` folder, then open **Notebook 2** for
per-class metrics / ROC-AUC / model complexity, and **Notebook 3** for
the ablation study + statistical significance test.
